# Language Model for the Shakespeare Dataset

The idea of this project is use text preprocessing tools and RNN models to build a character-level language model. Train the model on the works of Shakespeare and use the network to generate own text.

In [49]:
# Package Imports

import tensorflow as tf
import numpy as np
import json
import matplotlib.pyplot as plt
%matplotlib inline

##
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.data import Dataset



![imagen-shakespeare](data/shakespeare.png)

## The Shakespeare Dataset

We will work with a subset of the [Shakespeare dataset](http://shakespeare.mit.edu). It consists of a single text file with severeal excerpts concatenated together. The data is in a row text form and so far has not yet had any preprocessing. 

Therefore, the goal is construct an unsupervised character-level sequence model that can generate text according to a distribution learned from the dataset

### Loanding and Inspecting the Dataset

In [2]:
# Load text file into a string

with open('data/Shakespeare.txt', 'r', encoding='utf-8') as file:
    text = file.read()


In [3]:
#Printing 1000 letters of the string text

text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [4]:
# Creating a list of chunks of text
# We separate the sentences

text_chunks = text.split('.') 


In [5]:
# Looking the first 10 chunks
text_chunks[:10]


['First Citizen:\nBefore we proceed any further, hear me speak',
 '\n\nAll:\nSpeak, speak',
 '\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved',
 ' resolved',
 '\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people',
 "\n\nAll:\nWe know't, we know't",
 "\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price",
 "\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens",
 '\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good',
 '\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for t

Looking few chunks from the list in random order

In [6]:
num_samples = 10
idx = np.random.choice(len(text_chunks), num_samples, replace=False)
for chunck in np.array(text_chunks)[idx]:
    print(chunck)



KING RICHARD II:
'Fair cousin'? I am greater than a king:
For when I was a king, my flatterers
Were then but subjects; being now a subject,
I have a king here to my flatterer

I shall attend your leisure: but make haste;
The vaporous night approaches


KING LEWIS XI:
But, Warwick,
Thou and Oxford, with five thousand men,
Shall cross the seas, and bid false Edward battle;
And, as occasion serves, this noble queen
And prince shall follow with a fresh supply


LUCENTIO:
Mistress, what's your opinion of your sister?

BIANCA:
That, being mad herself, she's madly mated
 Good thou, save
me a piece of marchpane; and, as thou lovest me, let
the porter let in Susan Grindstone and Nell


KING RICHARD III:
Well, let that pass

We are lucky, boy; and to be so still requires
nothing but secrecy


DUKE VINCENTIO:
Hail to you, provost! so I think you are


SICINIUS:
Draw near, ye people

Thy husband in thy bosom there lies dead;
And Paris too


## Creating a character-level tokenizer

In [7]:

def create_character_tokenizer(list_of_strings):

    tokenizer = Tokenizer(
        num_words=None,
        filters='',
        lower= False,
        char_level = True,
        oov_token='<UNK>',
    )
    tokenizer.fit_on_texts(list_of_strings)
    return tokenizer




In [ ]:
#Getting the tokenizer

tokenizer = create_character_tokenizer(text_chunks)


In [19]:
tokenizer_config = tokenizer.get_config()
print(tokenizer_config)
print(tokenizer_config.keys())
print(tokenizer_config['word_counts'])

{'num_words': None, 'filters': '', 'lower': False, 'split': ' ', 'char_level': True, 'oov_token': '<UNK>', 'document_count': 7886, 'word_counts': '{"F": 1797, "i": 45537, "r": 48889, "s": 49696, "t": 67009, " ": 169892, "C": 3820, "z": 356, "e": 94611, "n": 48529, ":": 10316, "\\n": 40000, "B": 2761, "f": 15770, "o": 65798, "w": 17585, "p": 10808, "c": 15623, "d": 31358, "a": 55507, "y": 20448, "u": 26584, "h": 51310, ",": 19846, "m": 22243, "k": 7088, "A": 7819, "l": 33339, "S": 4523, "Y": 1718, "v": 7793, "?": 2462, "R": 4869, "M": 2840, "W": 3530, "\'": 6187, "L": 3876, "I": 11832, "N": 5079, "g": 13356, ";": 3628, "b": 11321, "!": 2172, "O": 5481, "j": 628, "V": 798, "-": 1897, "T": 7015, "H": 3068, "E": 6041, "U": 3313, "D": 2089, "P": 1641, "q": 609, "x": 529, "J": 320, "G": 2399, "K": 1584, "Q": 231, "&": 3, "Z": 198, "X": 112, "3": 27, "$": 1}', 'word_docs': '{"a": 7414, "p": 4562, "k": 3881, "y": 6301, "r": 7363, "o": 7610, "m": 6454, "i": 7187, ":": 6460, "u": 6580, "n": 7229

### Tokenize the text

In [20]:


def strings_to_sequences(tokenizer, list_of_strings):

    sentence_sequence = tokenizer.texts_to_sequences(list_of_strings)

    return sentence_sequence


In [21]:
seq_chunks = strings_to_sequences(tokenizer, text_chunks)

In [29]:
print(len(seq_chunks))
print(len(seq_chunks[0]))
print(seq_chunks[0])


7886
59
[50, 11, 9, 8, 4, 2, 38, 11, 4, 11, 58, 3, 10, 26, 12, 44, 3, 20, 5, 9, 3, 2, 19, 3, 2, 25, 9, 5, 21, 3, 3, 14, 2, 6, 10, 17, 2, 20, 15, 9, 4, 7, 3, 9, 18, 2, 7, 3, 6, 9, 2, 16, 3, 2, 8, 25, 3, 6, 29]


### Padding the encoded sequences and store them in a numpy array

In [31]:


def make_padded_dataset(sequence_chunks):

    sequence_padded = pad_sequences(sequence_chunks,
                                    maxlen=500,
                                    padding='pre',
                                    truncating='pre',
                                    value=0)

    return sequence_padded

In [32]:
#Padding the token sequence chunks and getting the numpy array

padded_sequences = make_padded_dataset(seq_chunks)



In [38]:
print(padded_sequences.shape)
print(padded_sequences[55])



(7886, 500)
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0

## Creating the model inputs and targets

The idea is build an RNN model. The model will receive a sequence of characters and predict the next character in the sequence. At training time, the model can be passed an input sequence, with the target sequence is shifted by one. 

For example, the expression `To be or not to be` appears in Shakespeare's play 'Hamlet'. Given input `To be or not to b`, the correct prediction is `o be or not to be`. The prediction is the same length as the input.

![image-example](data/rnn_example.png)

In [39]:


def create_inputs_and_targets(array_of_sequences):

    input_array = array_of_sequences[:, :-1]
    output_array = array_of_sequences[:, 1:]

    return (input_array, output_array)



In [40]:
# Input and output arrays

input_seq, target_seq = create_inputs_and_targets(padded_sequences)



### Preprocess Sequence array for Stateful RNN

The idea is build a RNN Language model to be stateful, so that the internal state of the RNN will be maintained across batches. 

In [41]:

#Batch size for training

batch_size = 32


In [42]:

#Preparing the input and output arrays for training stateful RNN

num_examples = input_seq.shape[0]

num_processed_examples = num_examples - (num_examples % batch_size)

input_seq = input_seq[:num_processed_examples]
target_seq = target_seq[:num_processed_examples]

steps = int(num_processed_examples / 32)

idx = np.empty((0,), dtype=np.int32)
for i in range(steps):
    idx = np.concatenate((idx, i + np.arange(0, num_processed_examples, steps)))


input_seq_stateful = input_seq[idx]
target_seq_stateful = target_seq[idx]


### Splitting the data into training an validation sets

In [44]:


num_train_examples = int(batch_size * ((0.8 * num_processed_examples) // batch_size))

input_train = input_seq_stateful[:num_train_examples]
target_train = target_seq_stateful[:num_train_examples]

input_valid = input_seq_stateful[num_train_examples:]
target_valid = target_seq_stateful[num_train_examples:]


### Creating training and validation Dataset objects

In [51]:

def make_Dataset(input_array, target_array, batch_size):

    dataset = Dataset.from_tensor_slices((input_array, target_array))

    dataset = dataset.batch( batch_size=batch_size, 
                            drop_remainder=True)

    return dataset


In [52]:

train_data = make_Dataset(input_train, target_train, batch_size)
valid_data = make_Dataset(input_valid, target_valid, batch_size)



## Building the recurrent neural network